# Project: Optimizer race 🏁
Race plain gradient descent, momentum and Adam down two famous loss landscapes: the **Rosenbrock banana** (a long curved valley) and a badly scaled **ellipse** (one direction 100× steeper). Watch why momentum and adaptive step sizes exist, and why Adam is everyone's default.

Topic: [[Gradient Descent]], [[Optimizers]] · guide note: [[Project - Optimizer Race]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing
def rosenbrock(w): x, y = w; return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2
def rosen_grad(w): x, y = w; return np.array([-2 * (1 - x) - 400 * x * (y - x ** 2), 200 * (y - x ** 2)])
def ellipse(w): return 0.5 * (w[0] ** 2 + 100 * w[1] ** 2)
def ellipse_grad(w): return np.array([w[0], 100 * w[1]])

## 1. Momentum (heavy ball)
Keep a running velocity: `v ← β·v + g`, then `w ← w − lr·v`. Return `(w, v)`.

In [ ]:
def momentum_step(w, v, g, lr, beta=0.9):
    # TODO 1: update velocity, then weights
    raise NotImplementedError  # TODO 1

## 2. Adam\n$m\leftarrow\beta_1m+(1-\beta_1)g$, $v\leftarrow\beta_2v+(1-\beta_2)g^2$, bias-correct with $1-\beta^t$ ($t$ starts at 1), then $w\leftarrow w-\text{lr}\,\hat m/(\sqrt{\hat v}+\epsilon)$. Return `(w, m, v)`.

In [ ]:
def adam_step(w, m, v, g, t, lr, b1=0.9, b2=0.999, eps=1e-8):
    # TODO 2: moment estimates, bias correction, update
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_opt():
    w, v = momentum_step(np.array([1.0]), np.array([2.0]), np.array([1.0]), lr=0.1)
    w2, m2, v2 = adam_step(np.array([0.0]), np.zeros(1), np.zeros(1), np.array([4.0]), 1, lr=0.1)
    return np.allclose(v, 2.8) and np.allclose(w, 0.72) and np.allclose(w2, -0.1, atol=1e-6) and np.allclose(m2, 0.4) and np.allclose(v2, 0.016)
check("optimizers", _t_opt, "Momentum: v = 0.9·2 + 1 = 2.8, w = 1 − 0.1·2.8. Adam's first step has size ≈ lr whatever the gradient's scale.")

def race(f, grad, start, steps, lrs):
    paths = {}
    w = np.array(start, float); p = [w.copy()]
    for _ in range(steps): w = w - lrs["GD"] * grad(w); p.append(w.copy())
    paths["GD"] = np.array(p)
    w, v = np.array(start, float), np.zeros(2); p = [w.copy()]
    for _ in range(steps): w, v = momentum_step(w, v, grad(w), lrs["momentum"]); p.append(w.copy())
    paths["momentum"] = np.array(p)
    w, m, v = np.array(start, float), np.zeros(2), np.zeros(2); p = [w.copy()]
    for t in range(1, steps + 1): w, m, v = adam_step(w, m, v, grad(w), t, lrs["Adam"]); p.append(w.copy())
    paths["Adam"] = np.array(p)
    return paths

final = {}
if ready("optimizers"):
    for name, f, g, start, lim, lrs in [("Rosenbrock", rosenbrock, rosen_grad, (-1.5, 2.0), ((-2, 2), (-1, 3)), {"GD": 1e-3, "momentum": 1e-4, "Adam": 0.02}),
                                        ("ellipse 1:100", ellipse, ellipse_grad, (9.0, 1.0), ((-10, 10), (-1.5, 1.5)), {"GD": 0.019, "momentum": 0.005, "Adam": 0.3})]:
        paths = race(f, g, start, 2000, lrs)
        xs, ys = np.meshgrid(np.linspace(*lim[0], 200), np.linspace(*lim[1], 200))
        zz = np.array([[f((a, b_)) for a in xs[0]] for b_ in ys[:, 0]])
        plt.figure(figsize=(6, 4)); plt.contour(xs, ys, np.log1p(zz), 30, cmap="Greys")
        for k, p in paths.items(): plt.plot(*p.T, ".-", ms=2, lw=1, label=f"{k}: loss {f(p[-1]):.1e}"); final[(name, k)] = f(p[-1])
        plt.legend(); plt.title(f"{name}, 2000 steps"); plt.show()
    check("adaptive wins", lambda: final[("Rosenbrock", "Adam")] < final[("Rosenbrock", "GD")] and final[("ellipse 1:100", "Adam")] < final[("ellipse 1:100", "GD")],
          "Adam should end lower than plain GD on both landscapes.")

**Why:** GD's learning rate is capped by the *steepest* direction (lr < 2/100 on the ellipse), so it crawls along the flat one. Momentum accumulates speed along consistent directions; Adam rescales each coordinate by its own gradient size.

<details><summary>▶ Solution</summary>

```python
def momentum_step(w, v, g, lr, beta=0.9):
    v = beta * v + g
    return w - lr * v, v
```

```python
def adam_step(w, m, v, g, t, lr, b1=0.9, b2=0.999, eps=1e-8):
    m = b1 * m + (1 - b1) * g
    v = b2 * v + (1 - b2) * g ** 2
    m_hat = m / (1 - b1 ** t)
    v_hat = v / (1 - b2 ** t)
    return w - lr * m_hat / (np.sqrt(v_hat) + eps), m, v
```
</details>

## Stretch goals
- Set GD's learning rate on the ellipse to 0.021 (> 2/100). What happens and why?
- Add Nesterov momentum and RMSProp to the race.
- Add noise to the gradient (SGD) and see which optimizer copes best.